<a href="https://colab.research.google.com/github/kshathishka/Healthinsurancepredictor/blob/main/hc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Task 3: Hierarchical Clustering on Textual Dataset

This task performs hierarchical clustering on a sample textual dataset. It includes:
- TF-IDF Vectorization
- Calculation of pairwise distances using Euclidean, Cosine, and Manhattan metrics
- Plotting dendrograms for visualization
- Printing cluster assignments for `k=3` clusters

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist, squareform

# ─── Sample Dataset ───────────────────────────────────────────────────────────
documents = [
    "machine learning is a subset of artificial intelligence",
    "deep learning uses neural networks for pattern recognition",
    "natural language processing deals with text and speech",
    "computer vision helps machines understand images",
    "reinforcement learning trains agents through rewards",
    "supervised learning uses labeled data for training",
    "unsupervised learning finds hidden patterns in data",
    "text classification assigns categories to documents",
    "sentiment analysis determines opinion in text",
    "speech recognition converts spoken language to text",
    "image recognition identifies objects in images",
    "object detection locates objects within images",
]

labels = [
    "ML-General", "Deep-Learning", "NLP", "CV",
    "RL", "Supervised", "Unsupervised", "Text-Class",
    "Sentiment", "Speech", "Image-Recog", "Object-Det"
]

print("Sample documents and labels loaded.")

### Step 1: TF-IDF Vectorization

We convert the text documents into numerical TF-IDF (Term Frequency-Inverse Document Frequency) vectors. This process quantifies the importance of words in a document relative to a corpus.

In [ ]:
# ─── Step 1: TF-IDF Vectorization ─────────────────────────────────────────────
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(documents).toarray()
print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape}")

### Step 2: Compute Pairwise Distance Matrices

We calculate the pairwise distances between the TF-IDF vectors using three different metrics:
- **Euclidean distance**: The straight-line distance between two points in Euclidean space.
- **Cosine distance**: Measures the cosine of the angle between two vectors, indicating their orientation similarity. Often used for text data after normalization.
- **Manhattan (City-block) distance**: The sum of the absolute differences of their Cartesian coordinates.

In [ ]:
# ─── Step 2: Compute Pairwise Distance Matrices ───────────────────────────────
# Euclidean distance
dist_euclidean = pdist(tfidf_matrix, metric='euclidean')

# Cosine distance
tfidf_normalized = normalize(tfidf_matrix)
dist_cosine = pdist(tfidf_normalized, metric='cosine')

# Manhattan (City-block) distance
dist_manhattan = pdist(tfidf_matrix, metric='cityblock')

print("Pairwise distance matrices computed.")

### Distance Configuration

Defining configurations for each distance metric, specifying the linkage method to be used for hierarchical clustering. Different linkage methods can yield different dendrogram structures.

In [ ]:
distance_configs = [
    ("Euclidean",  dist_euclidean,  "ward"),
    ("Cosine",     dist_cosine,     "average"),
    ("Manhattan",  dist_manhattan,  "complete"),
]

print("Distance configurations set up.")

### Step 3: Plot Dendrograms

Dendrograms visually represent the hierarchical clustering process. Each leaf represents a data point, and branches represent clusters, with their length indicating the distance at which clusters are merged. We will plot dendrograms for each distance metric.

In [ ]:
# ─── Step 3: Plot Dendrograms ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 7))
fig.suptitle("Hierarchical Clustering – Different Distance Metrics", fontsize=16, fontweight='bold')

for ax, (metric_name, dist_vec, method) in zip(axes, distance_configs):
    Z = linkage(dist_vec, method=method)
    dendrogram(
        Z,
        labels=labels,
        ax=ax,
        leaf_rotation=45,
        leaf_font_size=9,
        color_threshold=0.7 * max(Z[:, 2]),
    )
    ax.set_title(f"{metric_name} Distance\n(linkage: {method})", fontsize=12)
    ax.set_xlabel("Documents")
    ax.set_ylabel("Distance")

plt.tight_layout()
plt.savefig("/mnt/user-data/outputs/task3_dendrograms.png", dpi=150, bbox_inches='tight')
plt.show()
print("Dendrograms saved.")

### Step 4: Print Cluster Assignments

Finally, we will determine and print the cluster assignments for each document, assuming `k=3` clusters. This step helps in understanding which documents are grouped together based on the chosen distance metric and linkage method.

In [ ]:
# ─── Step 4: Print Cluster Assignments ────────────────────────────────────────
print("\n" + "="*60)
print("CLUSTER ASSIGNMENTS (k=3 clusters)")
print("="*60)

for metric_name, dist_vec, method in distance_configs:
    Z = linkage(dist_vec, method=method)
    clusters = fcluster(Z, t=3, criterion='maxclust')
    print(f"\n{metric_name} Distance:")
    for doc_label, cluster_id in zip(labels, clusters):
        print(f"  Cluster {cluster_id}: {doc_label}")